# F1 Pit Stop Prediction — Baseline EDA & Model
Kaggle Playground Series S6E5

## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

## Load Data

In [ ]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head()

## Explore

In [ ]:
train.info()
train.describe()

In [ ]:
print("Nulls:")
print(train.isnull().sum())

print("\nTarget distribution:")
print(train["PitNextLap"].value_counts(normalize=True))

## Prepare Features

In [ ]:
X = train.drop(columns=["PitNextLap"])
y = train["PitNextLap"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Valid: {X_valid.shape}")

## Train CatBoost

In [ ]:
categorical_features = ["Driver", "Compound", "Race"]

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid)
)

## Evaluate

In [ ]:
preds = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, preds)
print(f"Validation AUC: {auc:.6f}")

## Feature Importance

In [11]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

print(importance_df.to_string(index=False))

               Feature  Importance
                  Year   51.707691
                 Stint   14.471439
              TyreLife    9.863802
         LapTime_Delta    5.230705
          RaceProgress    4.014833
                  Race    3.657780
              Compound    3.453658
       Position_Change    1.579827
             LapNumber    1.275281
Cumulative_Degradation    1.054365
                Driver    0.970527
              Position    0.969339
               PitStop    0.888936
           LapTime (s)    0.861818
                    id    0.000000


In [12]:
test_preds = model.predict_proba(test)[:, 1]
print(test_preds[:10])

[0.00507831 0.00811448 0.00620564 0.08471492 0.71596929 0.18958102
 0.00483336 0.01215771 0.07585526 0.0031727 ]


In [14]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})
submission.to_csv("submission.csv", index=False)